# 🧠 Subject Training: FILE_TRANSFER

## Module: receive_file.py

In [ ]:
import socket


def main():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    host = socket.gethostname()
    port = 12312

    sock.connect((host, port))
    sock.send(b"Hello server!")

    with open("Received_file", "wb") as out_file:
        print("File opened")
        print("Receiving data...")
        while True:
            data = sock.recv(1024)
            if not data:
                break
            out_file.write(data)

    print("Successfully received the file")
    sock.close()
    print("Connection closed")


if __name__ == "__main__":
    main()


## Module: send_file.py

In [ ]:
def send_file(filename: str = "mytext.txt", testing: bool = False) -> None:
    import socket

    port = 12312  # Reserve a port for your service.
    sock = socket.socket()  # Create a socket object
    host = socket.gethostname()  # Get local machine name
    sock.bind((host, port))  # Bind to the port
    sock.listen(5)  # Now wait for client connection.

    print("Server listening....")

    while True:
        conn, addr = sock.accept()  # Establish connection with client.
        print(f"Got connection from {addr}")
        data = conn.recv(1024)
        print(f"Server received: {data = }")

        with open(filename, "rb") as in_file:
            data = in_file.read(1024)
            while data:
                conn.send(data)
                print(f"Sent {data!r}")
                data = in_file.read(1024)

        print("Done sending")
        conn.close()
        if testing:  # Allow the test to complete
            break

    sock.shutdown(1)
    sock.close()


if __name__ == "__main__":
    send_file()


## Module: test_send_file.py

In [ ]:
from unittest.mock import Mock, patch

from file_transfer.send_file import send_file


@patch("socket.socket")
@patch("builtins.open")
def test_send_file_running_as_expected(file, sock):
    # ===== initialization =====
    conn = Mock()
    sock.return_value.accept.return_value = conn, Mock()
    f = iter([1, None])
    file.return_value.__enter__.return_value.read.side_effect = lambda _: next(f)

    # ===== invoke =====
    send_file(filename="mytext.txt", testing=True)

    # ===== ensurance =====
    sock.assert_called_once()
    sock.return_value.bind.assert_called_once()
    sock.return_value.listen.assert_called_once()
    sock.return_value.accept.assert_called_once()
    conn.recv.assert_called_once()

    file.return_value.__enter__.assert_called_once()
    file.return_value.__enter__.return_value.read.assert_called()

    conn.send.assert_called_once()
    conn.close.assert_called_once()
    sock.return_value.shutdown.assert_called_once()
    sock.return_value.close.assert_called_once()
